In [ ]:
import numpy as np
import polars as pl # Polars
import plotly.graph_objects as go # 그래프 오브젝트 (익스프레스 말고)
import plotly.io as pio # 테마 전역설정
import plotly.express as px
from plotly.subplots import make_subplots # 서브플롯

In [ ]:
pio.templates.default = "plotly_white"
pio.templates[pio.templates.default]["layout"]["font"] = {"family": "Nanumsquare_ac", "size": 16}
pio.templates["plotly_white"]["layout"]["colorway"] = px.colors.sequential.algae

In [ ]:
clinvar_df = pl.read_csv('data/clinvar_20260404_analysis.csv', infer_schema_length=0)

In [ ]:
clinvar_df

# CLNSIG으로 묶기
- 흐름은 같고 툴만 다른거라 전처리는 다른 코드에서 했음

In [ ]:
clnsig_grp = clinvar_df.group_by('CLNSIG_Group').agg(
    pl.col('CLNSIG').count().alias("Total")
).sort("Total", descending=True) # 정렬을~ 돌려다아아아오~

- alias: 야 이거 묶은거 별칭 이걸로 해줘
- SQL 하신 분들은 뭔지 아실걸요?

## Plotly (bar chart)

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(x = clnsig_grp['CLNSIG_Group'], y = clnsig_grp['Total'], marker_color = px.colors.sequential.algae,text = clnsig_grp['Total'])
)

fig.update_layout(
    width = 1200, height = 800,
    xaxis=dict(title='ClinVar Significance Group'),
    yaxis=dict(title='Count'),
    title = "ClinVar Group Distribution",
    margin = dict(t=50, l=10, r=10, b=10)
)

# fig.write_image('group_distributuion.png')
fig.show()

## 트리이맵

In [ ]:
fig = px.treemap(
    clnsig_grp,
    path = ['CLNSIG_Group'],
    values = 'Total',
    color = 'Total',
    color_continuous_scale = 'algae',
    title = "ClinVar Significance Hierarchy"
)

# text_auto 대신 update_traces를 사용하여 블록 안에 텍스트를 강제로 넣습니다.
fig.update_traces(
    # label(그룹명)과 value(숫자)를 함께 표시하라는 명령입니다.
    textinfo = "label+value",
    # 숫자 포맷을 지정합니다 (천 단위 콤마: ,d)
    texttemplate = "%{label}<br>%{value:,d}",
    # 텍스트가 블록 크기에 맞게 조절되도록 설정
    textfont_size = 20,
    insidetextfont_family = "NanumSquare_ac" # 설정하신 폰트 적용
)

fig.update_layout(
    width = 1200,
    height = 800,
    margin = dict(t=50, l=10, r=10, b=10)
)

# fig.write_image('group_distributuion_heatmap.png')
fig.show()

- 개인적으로 cividis랑 트리맵은 궁합이 안 맞는 것 같다.

In [ ]:
clnsig_grp = clinvar_df.group_by(['CLNSIG_Group','CLNVC']).agg(
    pl.col('CLNSIG').count().alias("Total")
).sort("Total", descending=True) # 정렬을~ 돌려다아아아오~

In [ ]:
fig = px.treemap(
    clnsig_grp,
    path = ['CLNSIG_Group','CLNVC'],
    values = 'Total',
    color = 'CLNSIG_Group',
    color_continuous_scale = 'algae',
    title = "ClinVar Significance Hierarchy"
)

# text_auto 대신 update_traces를 사용하여 블록 안에 텍스트를 강제로 넣습니다.
fig.update_traces(
    # label(그룹명)과 value(숫자)를 함께 표시하라는 명령입니다.
    textinfo = "label+value",
    # 숫자 포맷을 지정합니다 (천 단위 콤마: ,d)
    texttemplate = "%{label}<br>%{value:,d}",
    # 텍스트가 블록 크기에 맞게 조절되도록 설정
    textfont_size = 20,
    insidetextfont_family = "NanumSquare_ac" # 설정하신 폰트 적용
)

fig.update_layout(
    width = 1200,
    height = 800,
    margin = dict(t=50, l=10, r=10, b=10)
)

# fig.write_image('group_distributuion_heatmap_2.png')
fig.show()

## 파이촤트

In [ ]:
fig = go.Figure()

fig.add_trace(go.Pie(
    labels = clnsig_grp['CLNSIG_Group'],
    values = clnsig_grp['Total'],
    textinfo = 'label+percent', # 이름과 퍼센트 동시에 표시
    insidetextorientation = 'radial',
    marker = dict(colors=px.colors.qualitative.Vivid) # 전역 설정 색감 유지
))

fig.update_layout(
    title_text = "ClinVar Significance Proportion",
    width = 1200,
    height = 800,
    margin = dict(t=50, l=10, r=10, b=10)
)

# fig.write_image('group_distributuion_pie.png')
fig.show()

# CLNSIG-Chromosome

In [ ]:
clnsig_grp = clinvar_df.group_by(['CLNSIG_Group', 'CHROM_Type']).agg(
    pl.col('CLNSIG').count().alias("Total")
).sort('CLNSIG_Group')

In [ ]:
clnsig_grp

In [ ]:
# 1. 사용할 팔레트를 가져옵니다
palette = px.colors.qualitative.Vivid

fig = go.Figure()

# 버블 크기 기준 설정
max_total = clnsig_grp['Total'].max()
sizeref = 2. * max_total / (100**2)

# 2. enumerate를 써서 순서(i)에 맞는 색상을 하나씩 배정합니다
for i, group_name in enumerate(clnsig_grp['CLNSIG_Group'].unique()):
    curr_df = clnsig_grp.filter(pl.col('CLNSIG_Group') == group_name)

    # 팔레트의 색상을 그룹 순서대로 하나씩 고릅니다
    # (팔레트보다 그룹이 많을 경우를 대비해 % 연산 사용)
    assigned_color = palette[i % len(palette)]

    fig.add_trace(go.Scatter(
        x = curr_df['CLNSIG_Group'],
        y = curr_df['CHROM_Type'],
        name = group_name,
        mode = 'markers+text',
        marker = dict(
            size = curr_df['Total'], # 여기 Total로 쓰셔야 버블 크기가 데이터 반영합니다!
            sizemode = 'area',
            sizeref = sizeref,
            sizemin = 10,
            # 핵심: 이 그룹에 배정된 '단 하나의 색상'을 넣어줍니다
            color = assigned_color
        ),
        text = curr_df['Total'],
        texttemplate = "%{x}<br>%{text:,d}",
        textposition = "top right",
        textfont = dict(family="NanumSquare_ac", size=12)
    ))

fig.update_layout(
    width = 1200,
    height = 800,
    title = "ClinVar Significance by Group (Color by Group)",
    showlegend = True,
    template = "plotly_white"
)

# fig.write_image('cln-chr bubble.png')
fig.show()

- 이런 차이는 씨본이 참 잘 나타내는듯... 아닌가? 씨본에는 버블차트 없죠?

### 언노운 나가주세요

In [ ]:
clnsig_grp = clinvar_df.group_by(['CLNSIG_Group', 'CHROM_Type']).agg(
    pl.col('CLNSIG').count().alias("Total")
).sort('CLNSIG_Group').filter(pl.col('CHROM_Type') != 'Unknown')

In [ ]:
# 1. 사용할 팔레트를 가져옵니다
palette = px.colors.qualitative.Vivid

fig = go.Figure()

# 버블 크기 기준 설정
max_total = clnsig_grp['Total'].max()
sizeref = 2. * max_total / (100**2)

# 2. enumerate를 써서 순서(i)에 맞는 색상을 하나씩 배정합니다
for i, group_name in enumerate(clnsig_grp['CLNSIG_Group'].unique()):
    curr_df = clnsig_grp.filter(pl.col('CLNSIG_Group') == group_name)

    # 팔레트의 색상을 그룹 순서대로 하나씩 고릅니다
    # (팔레트보다 그룹이 많을 경우를 대비해 % 연산 사용)
    assigned_color = palette[i % len(palette)]

    fig.add_trace(go.Scatter(
        x = curr_df['CLNSIG_Group'],
        y = curr_df['CHROM_Type'],
        name = group_name,
        mode = 'markers+text',
        marker = dict(
            size = curr_df['Total'], # 여기 Total로 쓰셔야 버블 크기가 데이터 반영합니다!
            sizemode = 'area',
            sizeref = sizeref,
            sizemin = 10,
            # 핵심: 이 그룹에 배정된 '단 하나의 색상'을 넣어줍니다
            color = assigned_color
        ),
        text = curr_df['Total'],
        texttemplate = "%{x}<br>%{text:,d}",
        textposition = "top right",
        textfont = dict(family="NanumSquare_ac", size=12)
    ))

fig.update_layout(
    width = 1200,
    height = 800,
    title = "ClinVar Significance by Group (Color by Group)",
    showlegend = True,
    template = "plotly_white"
)

# fig.write_image('cln-chr bubble_no unknown.png')
fig.show()

## 염색체별 비율 볼 수 있어요?

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(x = clnsig_grp['CHROM_Type'], y = clnsig_grp['Total'], marker_color = px.colors.sequential.algae, text = clnsig_grp['CLNSIG_Group'])
)

fig.update_layout(
    width = 1200, height = 800,
    xaxis=dict(title='Chromosome Type'),
    yaxis=dict(title='Count'),
    title = "ClinVar Group Distribution",
    margin = dict(t=50, l=10, r=10, b=10)
)

# fig.write_image('group_distributuion.png')
fig.show()

- 어... 이거 볼 수는 있는데요... 이거 이대로 포폴에 내면 안되는거 아시죠?

### 서브플롯

In [ ]:
# 1. 데이터에 있는 실제 값 순서 (가출 방지용)
real_values = ["Sex_Chrom", "Autosome", "Mitochondria"]

# 2. 그래프 위에 표시하고 싶은 예쁜 제목들 (순서 일치 필수!)
display_titles = ["Sex Chromosome", "Autosome", "Mitochondria"]

# 3. 서브플롯 생성
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=display_titles, # 여기서 예쁜 제목 적용!
    horizontal_spacing=0.08
)

# 4. 반복문은 실제 값(real_values)으로 돌립니다
for i, real_val in enumerate(real_values):
    curr_df = clnsig_grp.filter(pl.col('CHROM_Type') == real_val)

    # 내림차순 정렬해서 보기 좋게
    curr_df = curr_df.sort("Total", descending=True)

    fig.add_trace(
        go.Bar(
            x = curr_df['CLNSIG_Group'],
            y = curr_df['Total'],
            text = curr_df['Total'],
            texttemplate = '%{y:,}',
            textposition = 'outside',
            # 색깔은 이미 설정한 대로!
            marker_color = px.colors.sequential.algae[-(i*3+1)]
        ),
        row = 1, col = i + 1
    )

# 5. 레이아웃 정리
fig.update_layout(
    height=600,
    width=1250,
    title_text="ClinVar Distribution by Chromosome Type",
    showlegend=False,
    margin=dict(t=100, l=20, r=20, b=50), # 제목 공간 확보 위해 t(top) 늘림
)

# fig.write_image('group_distributuion_subplot.png')
fig.show()

- 서브플롯 박으셔도 되고

### 누적 막대 그래프

In [ ]:
# 1. 비율(Percentage) 계산 (Polars 활용)
clnsig_perc = clnsig_grp.with_columns(
    (pl.col("Total") / pl.col("Total").sum().over("CHROM_Type") * 100).alias("Percentage")
)

# 2. Plotly Express로 그리기 (비율 차트는 PX가 압도적으로 편합니다)
fig = px.bar(
    clnsig_perc,
    x = "CHROM_Type",
    y = "Percentage",
    color = "CLNSIG_Group", # 그룹별 색상
    text = "Percentage",
    title = "ClinVar Significance Proportion (100% Stacked)",
    color_discrete_sequence = px.colors.qualitative.Vivid # 아까 쓰신 Prism!
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside')
fig.update_layout(width=1200, height=800, barmode='stack')

# fig.write_image('group_distributuion_stack.png')
fig.show()

- 개인적으로는 서브플롯쪽이 더 낫다고 생각합니다... 누적 막대 그래프는 Risk/Other가 안보임..

# Pathogenic만 나와주세요

In [ ]:
pathogenic_df = clinvar_df.filter(pl.col('CLNSIG_Group') == 'Pathogenic')

In [ ]:
pathogenic_df

## 돌연변이예아아

In [ ]:
clnvc_grp = pathogenic_df.group_by('CLNVC').agg(
    pl.col('CLNSIG').count().alias("Total")
).sort('Total', descending=True)

In [ ]:
clnvc_grp

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(x = clnvc_grp['CLNVC'], y = clnvc_grp['Total'], marker_color = px.colors.sequential.algae, text = clnvc_grp['Total'])
)

fig.update_layout(
    width = 1200, height = 800,
    xaxis=dict(title='Pathogenic Variant', tickangle = -90),
    yaxis=dict(title='Count'),
    title = "Pathogenic Variant Distribution",
    margin = dict(t=50, l=10, r=10, b=10)
)

# fig.write_image('pathogenic_distributuion.png')
fig.show()

### 트리매앱

In [ ]:
fig = px.treemap(
    clnvc_grp,
    path = ['CLNVC'],
    values = 'Total',
    color = 'Total',
    color_continuous_scale = 'algae',
    title = "Pathogenic Variant Distribution",
)

# text_auto 대신 update_traces를 사용하여 블록 안에 텍스트를 강제로 넣습니다.
fig.update_traces(
    # label(그룹명)과 value(숫자)를 함께 표시하라는 명령입니다.
    textinfo = "label+value",
    # 숫자 포맷을 지정합니다 (천 단위 콤마: ,d)
    texttemplate = "%{label}<br>%{value:,d}",
    # 텍스트가 블록 크기에 맞게 조절되도록 설정
    textfont_size = 20,
    insidetextfont_family = "NanumSquare_ac" # 설정하신 폰트 적용
)

fig.update_layout(
    width = 1200,
    height = 800,
    margin = dict(t=50, l=10, r=10, b=10)
)

# fig.write_image('pathigenic_distributuion_heatmap.png')
fig.show()

### 파이

In [ ]:
fig = go.Figure()

fig.add_trace(go.Pie(
    labels = clnvc_grp['CLNVC'],
    values = clnvc_grp['Total'],
    textinfo = 'label+percent', # 이름과 퍼센트 동시에 표시
    insidetextorientation = 'radial',
    marker = dict(colors=px.colors.qualitative.Vivid) # 전역 설정 색감 유지
))

fig.update_layout(
    title_text = "Pathogenic Variant Distribution",
    width = 1200,
    height = 800,
    margin = dict(t=50, l=20, r=20, b=10)
)

# fig.write_image('pathgenic_distributuion_pie.png')
fig.show()

- 얘도 처참하다... SNV가 너무 압**도적**임...

## 염색체 종류별

In [ ]:
clnvc_grp = pathogenic_df.group_by(['CHROM_Type','CLNVC']).agg(
    pl.col('CHROM_Type').count().alias("Total")
).sort('CHROM_Type')

In [ ]:
clnvc_grp

In [ ]:
# 1. 데이터에 있는 실제 값 순서 (가출 방지용)
real_values = ["Sex_Chrom", "Autosome", "Mitochondria"]

# 2. 그래프 위에 표시하고 싶은 예쁜 제목들 (순서 일치 필수!)
display_titles = ["Sex Chromosome", "Autosome", "Mitochondria"]

# 3. 서브플롯 생성
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=display_titles, # 여기서 예쁜 제목 적용!
    horizontal_spacing=0.08
)

# 4. 반복문은 실제 값(real_values)으로 돌립니다
for i, real_val in enumerate(real_values):
    curr_df = clnvc_grp.filter(pl.col('CHROM_Type') == real_val)

    # 내림차순 정렬해서 보기 좋게
    curr_df = curr_df.sort("Total", descending=True)

    fig.add_trace(
        go.Bar(
            x = curr_df['CLNVC'],
            y = curr_df['Total'],
            text = curr_df['Total'],
            texttemplate = '%{y:,}',
            textposition = 'outside',
            # 색깔은 이미 설정한 대로!
            marker_color = px.colors.sequential.algae[-(i*3+1)]
        ),
        row = 1, col = i + 1
    )

# 5. 레이아웃 정리
fig.update_layout(
    height=600,
    width=1250,
    title_text="ClinVar Distribution by Chromosome Type",
    showlegend=False,
    margin=dict(t=100, l=20, r=20, b=50), # 제목 공간 확보 위해 t(top) 늘림
)

# fig.write_image('pathgenic_chr_distributuion_subplot.png')
fig.show()

- 그냥 SNV가 압도적임

## 종류 말고 구체적으로 봅시다
- 근데 미토콘드리아는 하나뿐이구나...

In [ ]:
clnvc_grp = pathogenic_df.group_by(['CHROM','CLNVC']).agg(
    pl.col('CHROM').count().alias("Total")
).sort('CHROM')

In [ ]:
clnvc_grp

In [ ]:
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']

# 2. 비율 계산
clnsig_perc = clnvc_grp.with_columns(
    (pl.col("Total") / pl.col("Total").sum().over("CHROM") * 100).alias("Percentage")
)

# 3. Plotly Express로 그리기
fig = px.bar(
    clnsig_perc,
    x = "CHROM",
    y = "Percentage",
    color = "CLNVC",
    text = "Percentage",
    title = "CLNVC ratio by Chromosome",
    color_discrete_sequence = px.colors.qualitative.Vivid,
    # --- 이 부분이 마법의 한 줄입니다 ---
    category_orders = {"CHROM": chr_order}
    # ----------------------------------
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='inside',
    insidetextfont_family="NanumSquare_ac" # 설정하신 폰트 적용
)

fig.update_layout(
    width=1250,
    height=800,
    barmode='stack',
    xaxis=dict(title='Chromosome'),
    yaxis=dict(title='Percentage (%)'),
    margin=dict(t=100, l=20, r=20, b=50),
)

# fig.write_image('chr_ratio.png')
fig.show()

## 각 염색체별 변이 넘버원

In [ ]:
# 1. 염색체 정렬 순서 정의
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
# 순서대로 숫자를 매칭한 딕셔너리 생성 (1:0, 2:1, ..., X:22, Y:23...)
chr_map = {val: i for i, val in enumerate(chr_order)}

# 2. 데이터 집계 및 필터링
top_gene_df = (
    pathogenic_df
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.count('GENE_SYMBOL').alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 3. 정렬 로직 수정 (최신 Polars 버전 대응)
top_gene_df = (
    top_gene_df
    .with_columns(
        pl.col("CHROM").cast(pl.Utf8)
    )
    .with_columns(
        # replace 대신 replace_strict를 사용하고, 매핑 안 되는 값은 99로 처리합니다.
        pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx")
    )
    .sort("sort_idx")
)

In [ ]:
top_gene_df

In [ ]:
# 1. 변이 수 기준으로 데이터 정렬 (표 상단에 TOP 3가 오게 하려면 여기서 정렬)
# 현재 top_gene_df가 염색체 순서라면, 강조 로직을 수치 기준으로 적용해야 합니다.
top_gene_df = top_gene_df.sort("Count", descending=True)

# 2. 행별 배경색 결정 함수
def get_rank_color(i):
    if i == 0: return '#FFD700' # Gold (1위: BRCA2)
    if i == 1: return '#E5E4E2' # Platinum/Silver (2위: TTN)
    if i == 2: return '#CD7F32' # Bronze (3위: BRCA1)
    return 'white' if i % 2 == 0 else '#F9F9F9' # 나머지 가독성용 줄무늬

row_colors = [get_rank_color(i) for i in range(len(top_gene_df))]

# 3. 데이터 구성
header_values = ["<b>Chromosome</b>", "<b>Top Gene Symbol</b>", "<b>Variant Count</b>"]
cell_values = [
    top_gene_df["CHROM"],
    [f"<b>{g}</b>" if i < 3 else g for i, g in enumerate(top_gene_df["GENE_SYMBOL"])],
    [f"{c:,}" for c in top_gene_df["Count"]]
]

# 4. Table 생성
fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#2E4A3F', # 헤더는 짙은 Algae색 유지
        align = 'center',
        font = dict(color='white', size=16, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_colors] * 3,
        align = ['center', 'center', 'right'],
        # 글자색은 무조건 검은색 계열로 고정하여 가시성 확보
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 35,
        line_color = '#E5E5E5' # 셀 구분선 살짝 추가
    )
)])

fig.update_layout(
    title = "Top 3 Mutated Genes Highlighted",
    width = 750,
    height = 1010,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene.png')
fig.show()

In [ ]:
# 1. 염색체 정렬 순서 정의
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
# 순서대로 숫자를 매칭한 딕셔너리 생성 (1:0, 2:1, ..., X:22, Y:23...)
chr_map = {val: i for i, val in enumerate(chr_order)}

# 2. 데이터 집계 및 필터링
top_gene_df = (
    pathogenic_df
    .group_by(['CHROM', 'CLNVC','GENE_SYMBOL'])
    .agg(pl.count('GENE_SYMBOL').alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 3. 정렬 로직 수정 (최신 Polars 버전 대응)
top_gene_df = (
    top_gene_df
    .with_columns(
        pl.col("CHROM").cast(pl.Utf8)
    )
    .with_columns(
        # replace 대신 replace_strict를 사용하고, 매핑 안 되는 값은 99로 처리합니다.
        pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx")
    )
    .sort("sort_idx")
)

In [ ]:
# 1. 변이 수 기준으로 데이터 정렬 (표 상단에 TOP 3가 오게 하려면 여기서 정렬)
# 현재 top_gene_df가 염색체 순서라면, 강조 로직을 수치 기준으로 적용해야 합니다.
top_gene_df = top_gene_df.sort("Count", descending=True)

# 2. 행별 배경색 결정 함수
def get_rank_color(i):
    if i == 0: return '#FFD700' # Gold (1위: BRCA2)
    if i == 1: return '#E5E4E2' # Platinum/Silver (2위: TTN)
    if i == 2: return '#CD7F32' # Bronze (3위: BRCA1)
    return 'white' if i % 2 == 0 else '#F9F9F9' # 나머지 가독성용 줄무늬

row_colors = [get_rank_color(i) for i in range(len(top_gene_df))]

# 3. 데이터 구성
header_values = ["<b>Chromosome</b>", "<b>CLNVC</b>", "<b>Top Gene Symbol</b>", "<b>Variant Count</b>"]
cell_values = [
    top_gene_df["CHROM"],top_gene_df["CLNVC"],
    [f"<b>{g}</b>" if i < 3 else g for i, g in enumerate(top_gene_df["GENE_SYMBOL"])],
    [f"{c:,}" for c in top_gene_df["Count"]]
]

# 4. Table 생성
fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#2E4A3F', # 헤더는 짙은 Algae색 유지
        align = 'center',
        font = dict(color='white', size=16, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_colors] * 3,
        align = ['center', 'center', 'right'],
        # 글자색은 무조건 검은색 계열로 고정하여 가시성 확보
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 35,
        line_color = '#E5E5E5' # 셀 구분선 살짝 추가
    )
)])

fig.update_layout(
    title = "Top 3 Mutated Genes Highlighted",
    width = 750,
    height = 1010,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_clnvc.png')
fig.show()

## 가장 변이가 적은 유전자도 있나요?

In [ ]:
# 1. 염색체 정렬 순서 정의
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
# 순서대로 숫자를 매칭한 딕셔너리 생성 (1:0, 2:1, ..., X:22, Y:23...)
chr_map = {val: i for i, val in enumerate(chr_order)}

# 2. 데이터 집계 및 필터링
top_gene_df = (
    pathogenic_df
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.count('GENE_SYMBOL').alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").min().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 3. 정렬 로직 수정 (최신 Polars 버전 대응)
top_gene_df = (
    top_gene_df
    .with_columns(
        pl.col("CHROM").cast(pl.Utf8)
    )
    .with_columns(
        # replace 대신 replace_strict를 사용하고, 매핑 안 되는 값은 99로 처리합니다.
        pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx")
    )
    .sort("sort_idx")
)

In [ ]:
# 1. 데이터 구성 (top_gene_df가 이미 Count 정렬되어 있다고 가정)
header_values = ["<b>Chromosome</b>", "<b>Gene Symbol</b>", "<b>Variant Count</b>"]
cell_values = [
    top_gene_df["CHROM"],
    # TOP 3는 굵게, 나머지는 보통 (배경색 대신 텍스트로만 구분)
    [f"<b>{g}</b>" if i < 3 else g for i, g in enumerate(top_gene_df["GENE_SYMBOL"])],
    [f"{c:,}" for c in top_gene_df["Count"]]
]

# 2. 줄무늬(Zebra) 배경색 설정 (눈의 피로도를 줄여줍니다)
row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(top_gene_df))]

# 3. Table 생성
fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#2E4A3F', # 헤더만 포인트 컬러 유지
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac"),
        height = 35
    ),
    cells = dict(
        values = cell_values,
        # 화려한 배경색 빼고 깔끔하게 화이트/그레이 교차
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 32,
        line_color = '#E5E5E5' # 얇은 경계선으로 구분감만 제공
    )
)])

fig.update_layout(
    title = "Least Mutated Genes (ClinVar Pathogenic)",
    width = 750,
    height = 1010,
    margin = dict(t=80, l=20, r=20, b=20),
    template="plotly_white"
)
# fig.write_image('chr_gene_low.png')
fig.show()

- 얘네들은 그냥 변이되면 질병 수준이 아니라 생존을 못하거나 대를 못 이을 가능성이...

## 각 변이별 Pathogenic의 비율

In [ ]:
clnsig_grp = clinvar_df.group_by(['CLNSIG_Group','CLNVC']).agg(
    pl.col('CLNSIG').count().alias("Total")
).sort("CLNSIG_Group", descending=True).filter(pl.col('CLNSIG_Group') != "Unknown") # 정렬을~ 돌려다아아아오~

In [ ]:
clnsig_grp

In [ ]:
clnsig_grp['CLNSIG_Group'].unique()

In [ ]:
# 다섯개다...
# 1. 데이터에 있는 실제 값 순서 (가출 방지용)
real_values = clnsig_grp['CLNSIG_Group'].unique()

# 2. 그래프 위에 표시하고 싶은 예쁜 제목들 (순서 일치 필수!)
display_titles = ["Benign", "Risk/Other", "Pathgenic", "VUS"]

# 3. 서브플롯 생성
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=display_titles, # 여기서 예쁜 제목 적용!
    horizontal_spacing=0.08,
    vertical_spacing=0.2
)

# 4. 반복문은 실제 값(real_values)으로 돌립니다
for i, real_val in enumerate(real_values):
    # i가 0, 1, 2, 3일 때 row와 col을 계산
    # i // 2 는 0, 0, 1, 1 (행)
    # i % 2  는 0, 1, 0, 1 (열)
    curr_row = (i // 2) + 1
    curr_col = (i % 2) + 1

    curr_df = clnsig_grp.filter(pl.col('CLNSIG_Group') == real_val)
    curr_df = curr_df.sort("Total", descending=True)

    fig.add_trace(
        go.Bar(
            x = curr_df['CLNVC'],
            y = curr_df['Total'],
            text = curr_df['Total'],
            texttemplate = '%{y:,}',
            textposition = 'outside',
            marker_color = px.colors.sequential.algae[-(i*2+2)] # 색상 간격 살짝 조정,
        ),
        row = curr_row,
        col = curr_col
    )
# 5. 레이아웃 및 축 설정 (가장 중요)
fig.update_yaxes(
    # 막대 끝 숫자가 잘리지 않도록 축 밖으로 나가는 것을 허용
    cliponaxis=False,
    # 각 서브플롯의 데이터 최대값보다 30% 더 여유 있게 천장 높이기
    # matches=None 속성이 있어야 독립적으로 적용됩니다.
    selector=dict(type='yaxes')
)

# 반복문 밖에서 한 번에 모든 서브플롯의 Y축 범위를 넉넉하게 조정
for i in range(len(real_values)):
    curr_row = (i // 2) + 1
    curr_col = (i % 2) + 1

    # 해당 그룹의 최대값 찾기
    m_val = clnsig_grp.filter(pl.col('CLNSIG_Group') == real_values[i])['Total'].max()

    fig.update_yaxes(
        range=[0, m_val * 1.30], # 35% 여유 (숫자가 클수록 이 비율이 안전합니다)
        row=curr_row, col=curr_col
    )

fig.update_layout(
    height=1000,
    width=1250,
    title_text="ClinVar Distribution by Variant Type", # 제목도 수정 (CLNVC 기준이니까요!)
    margin=dict(t=150, l=50, r=50, b=80), # 상단 여백(t)을 150까지 확 늘리세요
    # 서브플롯 간 수직 간격을 더 벌립니다 (0.1 -> 0.2)
    grid={'rows': 2, 'columns': 2, 'pattern': 'independent'},
    showlegend=False
)

# 서브플롯 사이의 간격을 조절하는 가장 직접적인 파라미터
fig.update_layout(
    bargap=0.2, # 막대 사이 간격
    yaxis_tickformat=',', # Y축 숫자 콤마
)

# fig.write_image('clnvc_distributuion.png')
fig.show()

# 부록-각 변이별 랭킹 1위
- SNV는 위에 다 나와서 생략합니다

In [ ]:
pathogenic_df['CLNVC'].unique()

## Deletion

In [ ]:
# 1. Deletion만 필터링 후 염색체별 최다 발생 유전자 집계
deletion_top_df = (
    pathogenic_df
    .filter(pl.col("CLNVC").str.contains("Deletion")) # 'Deletion' 포함된 항목 필터링
    .filter(pl.col("GENE_SYMBOL").is_not_null())      # 유전자 이름 없는 것 제외
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.len().alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 2. 염색체 정렬 (1~22, X, Y, MT 순서)
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
chr_map = {val: i for i, val in enumerate(chr_order)}

deletion_top_df = (
    deletion_top_df
    .with_columns(pl.col("CHROM").cast(pl.Utf8))
    .with_columns(pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx"))
    .sort("sort_idx")
)

# 3. 표 그리기 (배경색 없이 깔끔한 버전)
header_values = ["<b>Chromosome</b>", "<b>Top Deletion Gene</b>", "<b>Count</b>"]
cell_values = [
    deletion_top_df["CHROM"],
    deletion_top_df["GENE_SYMBOL"],
    [f"{c:,}" for c in deletion_top_df["Count"]]
]

row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(deletion_top_df))]

fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#1E3D59', # Deletion은 약간 차분한 네이비 톤으로 변경
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 30
    )
)])

fig.update_layout(
    title = "Top 1 Genes for 'Deletion' by Chromosome",
    width = 750,
    height = 900,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_deletion.png')
fig.show()

## Duplication

In [ ]:
# 1. Deletion만 필터링 후 염색체별 최다 발생 유전자 집계
deletion_top_df = (
    pathogenic_df
    .filter(pl.col("CLNVC").str.contains("Duplication"))
    .filter(pl.col("GENE_SYMBOL").is_not_null())      # 유전자 이름 없는 것 제외
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.len().alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 2. 염색체 정렬 (1~22, X, Y, MT 순서)
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
chr_map = {val: i for i, val in enumerate(chr_order)}

deletion_top_df = (
    deletion_top_df
    .with_columns(pl.col("CHROM").cast(pl.Utf8))
    .with_columns(pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx"))
    .sort("sort_idx")
)

# 3. 표 그리기 (배경색 없이 깔끔한 버전)
header_values = ["<b>Chromosome</b>", "<b>Top Deletion Gene</b>", "<b>Count</b>"]
cell_values = [
    deletion_top_df["CHROM"],
    deletion_top_df["GENE_SYMBOL"],
    [f"{c:,}" for c in deletion_top_df["Count"]]
]

row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(deletion_top_df))]

fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#1E3D59', # Deletion은 약간 차분한 네이비 톤으로 변경
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 30
    )
)])

fig.update_layout(
    title = "Top 1 Genes for 'Duplication' by Chromosome",
    width = 750,
    height = 900,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_duplication.png')
fig.show()

## Insertion

In [ ]:
# 1. Deletion만 필터링 후 염색체별 최다 발생 유전자 집계
deletion_top_df = (
    pathogenic_df
    .filter(pl.col("CLNVC").str.contains("Insertion"))
    .filter(pl.col("GENE_SYMBOL").is_not_null())      # 유전자 이름 없는 것 제외
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.len().alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 2. 염색체 정렬 (1~22, X, Y, MT 순서)
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
chr_map = {val: i for i, val in enumerate(chr_order)}

deletion_top_df = (
    deletion_top_df
    .with_columns(pl.col("CHROM").cast(pl.Utf8))
    .with_columns(pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx"))
    .sort("sort_idx")
)

# 3. 표 그리기 (배경색 없이 깔끔한 버전)
header_values = ["<b>Chromosome</b>", "<b>Top Deletion Gene</b>", "<b>Count</b>"]
cell_values = [
    deletion_top_df["CHROM"],
    deletion_top_df["GENE_SYMBOL"],
    [f"{c:,}" for c in deletion_top_df["Count"]]
]

row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(deletion_top_df))]

fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#1E3D59', # Deletion은 약간 차분한 네이비 톤으로 변경
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 30
    )
)])

fig.update_layout(
    title = "Top 1 Genes for 'Insertion' by Chromosome",
    width = 750,
    height = 900,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_insertion.png')
fig.show()

## Indel

In [ ]:
# 1. Deletion만 필터링 후 염색체별 최다 발생 유전자 집계
deletion_top_df = (
    pathogenic_df
    .filter(pl.col("CLNVC").str.contains("Indel"))
    .filter(pl.col("GENE_SYMBOL").is_not_null())      # 유전자 이름 없는 것 제외
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.len().alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 2. 염색체 정렬 (1~22, X, Y, MT 순서)
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
chr_map = {val: i for i, val in enumerate(chr_order)}

deletion_top_df = (
    deletion_top_df
    .with_columns(pl.col("CHROM").cast(pl.Utf8))
    .with_columns(pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx"))
    .sort("sort_idx")
)

# 3. 표 그리기 (배경색 없이 깔끔한 버전)
header_values = ["<b>Chromosome</b>", "<b>Top Deletion Gene</b>", "<b>Count</b>"]
cell_values = [
    deletion_top_df["CHROM"],
    deletion_top_df["GENE_SYMBOL"],
    [f"{c:,}" for c in deletion_top_df["Count"]]
]

row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(deletion_top_df))]

fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#1E3D59', # Deletion은 약간 차분한 네이비 톤으로 변경
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 30
    )
)])

fig.update_layout(
    title = "Top 1 Genes for 'Indel' by Chromosome",
    width = 750,
    height = 830,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_indel.png')
fig.show()

## Inversion

In [ ]:
# 1. Deletion만 필터링 후 염색체별 최다 발생 유전자 집계
deletion_top_df = (
    pathogenic_df
    .filter(pl.col("CLNVC").str.contains("Inversion"))
    .filter(pl.col("GENE_SYMBOL").is_not_null())      # 유전자 이름 없는 것 제외
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.len().alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 2. 염색체 정렬 (1~22, X, Y, MT 순서)
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
chr_map = {val: i for i, val in enumerate(chr_order)}

deletion_top_df = (
    deletion_top_df
    .with_columns(pl.col("CHROM").cast(pl.Utf8))
    .with_columns(pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx"))
    .sort("sort_idx")
)

# 3. 표 그리기 (배경색 없이 깔끔한 버전)
header_values = ["<b>Chromosome</b>", "<b>Top Deletion Gene</b>", "<b>Count</b>"]
cell_values = [
    deletion_top_df["CHROM"],
    deletion_top_df["GENE_SYMBOL"],
    [f"{c:,}" for c in deletion_top_df["Count"]]
]

row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(deletion_top_df))]

fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#1E3D59', # Deletion은 약간 차분한 네이비 톤으로 변경
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 30
    )
)])

fig.update_layout(
    title = "Top 1 Genes for 'Inversion' by Chromosome",
    width = 750,
    height = 860,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_inversion.png')
fig.show()

## Microsatellite

In [ ]:
# 1. Deletion만 필터링 후 염색체별 최다 발생 유전자 집계
deletion_top_df = (
    pathogenic_df
    .filter(pl.col("CLNVC").str.contains("Microsatellite"))
    .filter(pl.col("GENE_SYMBOL").is_not_null())      # 유전자 이름 없는 것 제외
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.len().alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 2. 염색체 정렬 (1~22, X, Y, MT 순서)
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
chr_map = {val: i for i, val in enumerate(chr_order)}

deletion_top_df = (
    deletion_top_df
    .with_columns(pl.col("CHROM").cast(pl.Utf8))
    .with_columns(pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx"))
    .sort("sort_idx")
)

# 3. 표 그리기 (배경색 없이 깔끔한 버전)
header_values = ["<b>Chromosome</b>", "<b>Top Deletion Gene</b>", "<b>Count</b>"]
cell_values = [
    deletion_top_df["CHROM"],
    deletion_top_df["GENE_SYMBOL"],
    [f"{c:,}" for c in deletion_top_df["Count"]]
]

row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(deletion_top_df))]

fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#1E3D59', # Deletion은 약간 차분한 네이비 톤으로 변경
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 30
    )
)])

fig.update_layout(
    title = "Top 1 Genes for 'Microsatellite' by Chromosome",
    width = 750,
    height = 900,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_microsatellite.png')
fig.show()

## Variation

In [ ]:
# 1. Deletion만 필터링 후 염색체별 최다 발생 유전자 집계
deletion_top_df = (
    pathogenic_df
    .filter(pl.col("CLNVC").str.contains("Variation"))
    .filter(pl.col("GENE_SYMBOL").is_not_null())      # 유전자 이름 없는 것 제외
    .group_by(['CHROM', 'GENE_SYMBOL'])
    .agg(pl.len().alias("Count"))
    .filter(pl.col("Count") == pl.col("Count").max().over("CHROM"))
    .unique(subset=["CHROM"], keep="first")
)

# 2. 염색체 정렬 (1~22, X, Y, MT 순서)
chr_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT', 'M']
chr_map = {val: i for i, val in enumerate(chr_order)}

deletion_top_df = (
    deletion_top_df
    .with_columns(pl.col("CHROM").cast(pl.Utf8))
    .with_columns(pl.col("CHROM").replace_strict(chr_map, default=99).alias("sort_idx"))
    .sort("sort_idx")
)

# 3. 표 그리기 (배경색 없이 깔끔한 버전)
header_values = ["<b>Chromosome</b>", "<b>Top Deletion Gene</b>", "<b>Count</b>"]
cell_values = [
    deletion_top_df["CHROM"],
    deletion_top_df["GENE_SYMBOL"],
    [f"{c:,}" for c in deletion_top_df["Count"]]
]

row_stripes = ['white' if i % 2 == 0 else '#F8F9F9' for i in range(len(deletion_top_df))]

fig = go.Figure(data=[go.Table(
    columnwidth = [80, 150, 100],
    header = dict(
        values = header_values,
        fill_color = '#1E3D59', # Deletion은 약간 차분한 네이비 톤으로 변경
        align = 'center',
        font = dict(color='white', size=15, family="NanumSquare_ac")
    ),
    cells = dict(
        values = cell_values,
        fill_color = [row_stripes] * 3,
        align = ['center', 'center', 'right'],
        font = dict(color='#333', size=14, family="NanumSquare_ac"),
        height = 30
    )
)])

fig.update_layout(
    title = "Top 1 Genes for 'Variation' by Chromosome",
    width = 750,
    height = 250,
    margin = dict(t=80, l=20, r=20, b=20)
)

# fig.write_image('chr_gene_variation.png')
fig.show()